In [1]:
# ==========================================
# 1. ENVIRONMENT SETUP
# ==========================================
# Run this in a separate Colab cell first
!pip uninstall -y unsloth unsloth_zoo
!pip install --prefer-binary "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps --prefer-binary xformers trl peft accelerate bitsandbytes wandb huggingface_hub

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-f2zqyf_4/unsloth_862787fe96fc4d99b7b98ab3014197df
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-f2zqyf_4/unsloth_862787fe96fc4d99b7b98ab3014197df
  Resolved https://github.com/unslothai/unsloth.git to commit 0a54d001ec0f6d65cc766480c76588cd8370f5a0
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 43.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 111.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 428.0/428.0 kB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 126.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 20.7 MB/s eta 0:00:0

In [3]:
# ==========================================
# 2. DPO TRAINING & W&B TRACKING
# ==========================================
import torch
import wandb
import os
from unsloth import FastLanguageModel, PatchDPOTrainer
from trl import DPOTrainer, DPOConfig
from datasets import load_dataset
from huggingface_hub import notebook_login

# --- AUTHENTICATION ---
print("Log in to Weights & Biases:")
wandb.login() # Prompts for W&B API Key

print("\nLog in to Hugging Face (Must have WRITE access):")
notebook_login() # Prompts for Hugging Face Token

# Initialize W&B Project
wandb.init(project="dpo-alignment-flywheel", name="qwen-3b-length-debiased-dpo")

# --- CONFIGURATION ---
MODEL_ID = "nallaramu/deliberate-qwen-2.5-3b-reasoning"
HF_PUSH_ID = "nallaramu/deliberate-qwen-2.5-3b-dpo" # Where the final model goes

print(f"Loading Base SFT Model: {MODEL_ID} in 4-bit...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_ID,
    max_seq_length = 2048,
    load_in_4bit = True,
)

# Apply LoRA Adapters for the Active DPO Model
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

# --- LOAD DATASET ---
if not os.path.exists("dpo_preference_pairs.jsonl"):
    raise FileNotFoundError("CRITICAL: Please upload 'dpo_preference_pairs.jsonl' to Colab!")

# Load the 74 length-debiased preference pairs
dataset = load_dataset("json", data_files="dpo_preference_pairs.jsonl", split="train")

# --- UNSLOTH DPO OPTIMIZATION ---
# This patches the trainer to avoid the 2x VRAM Reference Model requirement
PatchDPOTrainer()

# --- DPO CONFIGURATION ---
dpo_args = DPOConfig(
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    warmup_steps = 5,
    max_steps = 50, # 50 steps is enough for our 74 high-quality pairs
    learning_rate = 5e-5, # Lower LR than SFT to prevent representation collapse
    fp16 = not torch.cuda.is_bf16_supported(),
    bf16 = torch.cuda.is_bf16_supported(),
    logging_steps = 1,
    optim = "adamw_8bit",
    output_dir = "dpo_outputs",

    # W&B Integration
    report_to = "wandb",

    # DPO Specifics
    beta = 0.1, # The KL penalty. 0.1 allows behavior change without destroying logic.
    max_length = 2048,
    max_prompt_length = 1024,
    remove_unused_columns = False # Required so 'prompt', 'chosen', 'rejected' map correctly
)

print("Initializing DPOTrainer...")
dpo_trainer = DPOTrainer(
    model = model,
    ref_model = None, # Unsloth dynamically handles the reference model
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = dpo_args,
)

print("Starting Direct Preference Optimization...")
dpo_trainer.train()

# --- SAVE & PUSH ALIGNED MODEL ---
print("Training Complete! Pushing to Hugging Face...")
# We save locally first
model.save_pretrained("dpo_aligned_adapter")
tokenizer.save_pretrained("dpo_aligned_adapter")

# Push to Hub with professional tags
model.push_to_hub(
    HF_PUSH_ID,
    tags=["dpo", "rlhf", "reasoning", "unsloth", "alignment"],
    commit_message="Initial release of DPO-aligned debiased reasoning model"
)
tokenizer.push_to_hub(HF_PUSH_ID)

wandb.finish()
print(f"🚀 Success! Model live at: https://huggingface.co/{HF_PUSH_ID}")

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


Log in to Weights & Biases:

Log in to Hugging Face (Must have WRITE access):


Loading Base SFT Model: nallaramu/deliberate-qwen-2.5-3b-reasoning in 4-bit...
==((====))==  Unsloth 2026.5.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

unsloth/Qwen2.5-3B-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpvua5shgm/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

train/epoch,▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇███
train/global_step,▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇████
train/grad_norm,▄▃ ▄▄▃▃▄▂▂▄▅▄▄▃ ▂▃▂▃▃▂▅▂▄▂▂▂▃▃▄▄▃▂▂▁▃▂▁█
train/learning_rate,▁▂▄▄▅█████▇▇▇▇▆▆▆▆▆▆▅▅▅▅▅▄▄▄▄▄▃▃▃▃▃▂▂▂▂▂
train/logits/chosen,▅▃▄▄▃▃▃▃▁▆▅▄▆▄▆▅▅▆▆▆▇▆█▇▆▆▇▅▅▄▄█▅▅▄▄▃▄▆▃
train/logits/rejected,▃▁▃▃▂▂▃▁▃▃▃▅▄▇▆▅▅▅▆▆▇▄█▇▆▆▆▅▆▅▄█▆▅▄▆▂▃▇▂
train/logps/chosen,▆▆▇▇▇█▇▇█▇█▇▇▇▇▆▆▆▇▇▇▆▄▄▅▅▅▆▆▆▅▁▅▅▆▆▇▆▅▅
train/logps/rejected,▆▆▅▆▆▆▇▆█▆▇▆▆▅▅▂▇▄▅▃▅▆▄▄▃▃▄▃▃▂▂▃▁▂▄▁▆▄▁▅
train/loss,▄▃█▄▆▄▃▃▃▂▃▃▅▃▃▁▄▁▂▁▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/rewards/accuracies,▄▄▃▃▃▅▃▄▁▅▄▅▃▅▆▅▇▅▇▅▅█▇▇█▇██▇▇▆█▇███▇██▅
+3,...


🚀 Success! Model live at: https://huggingface.co/nallaramu/deliberate-qwen-2.5-3b-dpo


In [4]:
import shutil
from google.colab import files

# Define the folder where you saved the model in Stage 2
adapter_folder = "dpo_aligned_adapter"
zip_filename = "dpo_aligned_adapter.zip"

print(f"Zipping {adapter_folder}...")
shutil.make_archive(adapter_folder, 'zip', adapter_folder)

print(f"Downloading {zip_filename} to your local machine...")
files.download(zip_filename)

Zipping dpo_aligned_adapter...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>